# NORDIC noise-estimation workbench

**Thesis (the reason this notebook exists):** the NORDIC / MP hard threshold is
derived for i.i.d. unit-variance noise. Its keep/kill boundary — the bulk edge
`σ(√m+√n)` — is *exactly right* **if and only if** the noise really is i.i.d. with
that σ. So the whole game is the noise model:

- **g-factor** makes the per-voxel variance uniform (the diagonal),
- **σ** sets the absolute floor (a *separate* scalar — median-normalizing g throws it away),
- prewhitening would make the noise truly i.i.d. (the off-diagonal; temporal is free —
  thermal is white by physics).

Get those right and the factor is 1, the hard threshold is principled, and patch
size / shrinkage-vs-hard are second-order. This notebook is a workbench for
(1) estimating g-factor with **2, 1, or 0 noise volumes**, (2) calibrating σ, and
(3) an *honest* look at what the denoising residual can and cannot tell us —
which motivates the multi-echo / reproducibility direction (Part 3).

In [ ]:
import os
import sys
from math import gamma, sqrt
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
from scipy.ndimage import gaussian_filter

# Find the repo root from wherever the notebook is launched (no __file__ in nb).
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fastfuncstuff").is_dir()), Path.cwd())
sys.path.insert(0, str(REPO))
from fastfuncstuff.stats.voxel_correlation import analytic_r_null, corr_histogram_distance

torch.manual_seed(0)
DEV = torch.device("cpu")
print("ready")

## Part 0 — the toolkit

Everything is compact and self-contained so you can edit it. Three g-factor
estimators (one per noise-volume regime), σ calibration, the finite-size null,
a compact hard-threshold LLR, and the residual referees.

In [ ]:
# ---- g-factor across regimes (all return a map normalized to median 1) ----
def c4(k):  # unbiases the sample std of k obs
    return sqrt(2 / (k - 1)) * gamma(k / 2) / gamma((k - 1) / 2) if k >= 2 else 1.0

def gf_temporal(noise_vols, fwhm=2.0):          # >=2 noise volumes (SAUNA's path)
    k = noise_vols.shape[-1]
    var = torch.var(noise_vols.real, -1) + torch.var(noise_vols.imag, -1)
    std = torch.from_numpy(gaussian_filter(torch.sqrt(var).numpy(), fwhm)) / c4(k)
    return std / std.median().clamp(min=1e-8)

def gf_spatial(noise_vol1, fwhm=3.0):           # 1 volume: spatial pooling replaces time-averaging
    p = noise_vol1.real ** 2 + noise_vol1.imag ** 2                 # E|n|^2 = g^2
    g = torch.sqrt(torch.from_numpy(gaussian_filter(p.numpy(), fwhm)))
    return g / g.median().clamp(min=1e-8)

def gf_difference(data_sig, fwhm=3.0):          # 0 volumes: thermal is white-in-time
    d = data_sig[..., 1:] - data_sig[..., :-1]                       # var(diff)/2 ~ g^2 where signal smooth
    v = (torch.var(d.real, -1) + torch.var(d.imag, -1)) / 2
    g = torch.sqrt(torch.from_numpy(gaussian_filter(v.numpy(), fwhm)))
    return g / g.median().clamp(min=1e-8)

def calibrate_sigma(gc_noise):                  # absolute floor: sqrt(E|z|^2) in g-corrected space
    return float(torch.sqrt((gc_noise.real ** 2 + gc_noise.imag ** 2).mean()))

In [ ]:
# ---- finite-size null (#1): top singular value of a pure-noise patch ----
def sim_null(m, n, sigma, n_trials=1500, alpha=0.05, seed=0):
    gen = torch.Generator().manual_seed(seed)
    tops = []
    for i in range(0, n_trials, 256):
        b = min(256, n_trials - i)
        z = (torch.randn(b, m, n, generator=gen) + 1j * torch.randn(b, m, n, generator=gen)) / sqrt(2)
        tops.append(torch.linalg.svdvals(z * sigma)[:, 0])
    tops = torch.cat(tops)
    return float(torch.quantile(tops, 1 - alpha)), sigma * (sqrt(m) + sqrt(n))  # (finite-size thr, asymptotic MP edge)

# ---- compact hard-threshold LLR in the g-corrected (white) space ----
def _starts(dim, w, step):
    if dim <= w: return [0]
    s = list(range(0, dim - w + 1, max(1, step)))
    if s[-1] != dim - w: s.append(dim - w)
    return s

def denoise_hard(gc, sigma, kernel=(7, 7, 6), overlap=2, alpha=0.05):
    nx, ny, nz, nt = gc.shape
    wx, wy, wz = (min(k, d) for k, d in zip(kernel, (nx, ny, nz)))
    thr, mp_edge = sim_null(wx * wy * wz, nt, sigma, alpha=alpha)
    recon = torch.zeros_like(gc); wt = torch.zeros(nx, ny, nz)
    for x0 in _starts(nx, wx, max(1, wx // overlap)):
        for y0 in _starts(ny, wy, max(1, wy // overlap)):
            for z0 in _starts(nz, wz, max(1, wz // overlap)):
                P = gc[x0:x0+wx, y0:y0+wy, z0:z0+wz, :].reshape(wx * wy * wz, nt)
                u, s, vh = torch.linalg.svd(P, full_matrices=False)
                s = torch.where(s >= thr, s, torch.zeros_like(s))
                recon[x0:x0+wx, y0:y0+wy, z0:z0+wz, :] += ((u * s) @ vh).reshape(wx, wy, wz, nt)
                wt[x0:x0+wx, y0:y0+wy, z0:z0+wz] += 1
    return recon / wt.clamp(min=1.0)[..., None], thr, mp_edge

In [ ]:
# ---- referees on the residual ----
def residual_whiteness(residual, idx, coords):
    ts = residual.reshape(-1, residual.shape[-1])[idx].abs()
    return corr_histogram_distance(ts, coords[idx], n_dist_bins=10)

def pooled_patch_topsv(residual, kernel=(7, 7, 6), overlap=2):
    """The most sensitive single-dataset referee: NORDIC's own test re-run on the
    residual, top/median singular-value ratio pooled over ALL patches (sqrt(N_patch))."""
    nx, ny, nz, nt = residual.shape
    wx, wy, wz = (min(k, d) for k, d in zip(kernel, (nx, ny, nz)))
    vals = []
    for x0 in _starts(nx, wx, wx // overlap):
        for y0 in _starts(ny, wy, wy // overlap):
            for z0 in _starts(nz, wz, wz // overlap):
                P = residual[x0:x0+wx, y0:y0+wy, z0:z0+wz, :].reshape(wx * wy * wz, nt).abs()
                P = P - P.mean(0, keepdim=True)
                sv = torch.linalg.svdvals(P)
                vals.append(float(sv[0] / sv.median()))
    v = torch.tensor(vals)
    return float(v.mean()), float(v.std() / sqrt(len(v)))

## Part 1 — Synthetic ground-truth lab

Complex data with a **known** g-factor (smooth gradient), a low-rank spatially-
smooth signal in a central block with **temporally-colored** (BOLD-like) time
courses whose amplitudes decay so the weak components sit *near the noise floor*,
and white thermal noise scaled by g. Ground truth lets us prove the thesis
directly.

In [ ]:
def make_synthetic(nx=32, ny=32, nz=6, nt=100, rank=4, sig_amp=10.0, gf_contrast=2.5, n_noise=3, seed=0):
    rng = np.random.default_rng(seed)
    g = gaussian_filter(np.broadcast_to(np.linspace(1, gf_contrast, nx)[:, None, None], (nx, ny, nz)).copy(), 2).astype(np.float32)
    T = nt + n_noise
    amps = sig_amp * np.geomspace(1.0, 0.06, rank)                    # weak components straddle the floor
    tc = np.stack([gaussian_filter(rng.standard_normal(nt), 3.0) for _ in range(rank)])  # BOLD-like (colored)
    tc = (tc / tc.std(1, keepdims=True)) * amps[:, None]
    loads = np.stack([gaussian_filter(rng.standard_normal((16, 16, nz)), 2.0) for _ in range(rank)])
    block_sig = np.einsum("rxyz,rt->xyzt", loads, tc)
    sig = np.zeros((nx, ny, nz, T), np.complex64); sig[8:24, 8:24, :, :nt] = block_sig
    w = (rng.standard_normal((nx, ny, nz, T)) + 1j * rng.standard_normal((nx, ny, nz, T))) / sqrt(2)
    data = torch.from_numpy((sig + w * g[..., None]).astype(np.complex64))
    clean = torch.from_numpy(sig[..., :nt].astype(np.complex64))
    return data, torch.from_numpy(g), (slice(8, 24), slice(8, 24)), n_noise, clean

data, g_true, block, n_noise, clean = make_synthetic()
nt = data.shape[-1] - n_noise
sig, noise_vols = data[..., :nt], data[..., nt:]
NULL = analytic_r_null(nt)["mean_abs_r"]
print("data", tuple(data.shape), " nt", nt, " analytic null |r|", round(NULL, 3))

### g-factor recovery across regimes (2 / 1 / 0 noise volumes)

In [ ]:
def gerr(gest, gtrue):
    a, b = gest.flatten(), (gtrue / gtrue.median()).flatten()
    return float((a - b).pow(2).mean().sqrt() / b.mean())

g2 = gf_temporal(noise_vols)          # >=2 vols
g1 = gf_spatial(noise_vols[..., 0])   # 1 vol
g0 = gf_difference(sig)               # 0 vols
for name, g in [("temporal (>=2 vol)", g2), ("spatial (1 vol)", g1), ("difference (0 vol)", g0)]:
    print(f"  {name:20s} rel-RMSE vs truth = {gerr(g, g_true):.4f}")

fig, ax = plt.subplots(1, 4, figsize=(13, 3))
mid = g_true.shape[2] // 2
for a, (t, m) in zip(ax, [("truth", g_true / g_true.median()), ("2 vol", g2), ("1 vol", g1), ("0 vol", g0)]):
    im = a.imshow(m[:, :, mid].numpy().T, vmin=0.6, vmax=1.8, cmap="viridis"); a.set_title(t); a.axis("off")
fig.colorbar(im, ax=ax, shrink=0.7); plt.show()

### The money plot: signal recovery vs σ peaks at the correct floor

Ground truth lets us measure recovery = mean per-voxel corr(denoised, true signal)
in the block. **It peaks exactly at the correctly-calibrated σ (multiplier 1.0):**
under-remove and noise is left; over-remove and signal is eaten. This is the whole
thesis in one curve — get g and σ right, the hard threshold lands right.

In [ ]:
def signal_recovery(recon_gc, clean_gc, block):
    b = (block[0], block[1], slice(None))
    d = recon_gc[b].reshape(-1, recon_gc.shape[-1]).abs(); c = clean_gc[b].reshape(-1, clean_gc.shape[-1]).abs()
    d = d - d.mean(1, keepdim=True); c = c - c.mean(1, keepdim=True)
    return float(((d * c).sum(1) / (d.norm(dim=1) * c.norm(dim=1)).clamp(min=1e-8)).mean())

nx, ny, nz = data.shape[:3]
ii, jj, kk = torch.meshgrid(torch.arange(nx), torch.arange(ny), torch.arange(nz), indexing="ij")
coords = torch.stack([ii.ravel(), jj.ravel(), kk.ravel()], 1).float()
bm = torch.zeros(nx, ny, nz, dtype=torch.bool); bm[block[0], block[1], :] = True
bidx = torch.nonzero(bm.reshape(-1)).squeeze(1)

gnorm = (g_true / g_true.median()).clamp(min=1e-6)
gc = sig / gnorm[..., None]; clean_gc = clean / gnorm[..., None]
sigma0 = calibrate_sigma(noise_vols / gnorm[..., None])
print("calibrated sigma (g-corrected space):", round(sigma0, 3))

mults = [0.7, 0.85, 1.0, 1.15, 1.3, 1.6, 2.0]
rec, meanr, pooled, pooled_se = [], [], [], []
for mlt in mults:
    recon, thr, mp = denoise_hard(gc, sigma0 * mlt)
    resid = gc - recon
    rec.append(signal_recovery(recon, clean_gc, block))
    meanr.append(residual_whiteness(resid, bidx, coords).mean_abs_r)
    m, se = pooled_patch_topsv(resid); pooled.append(m); pooled_se.append(se)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(mults, rec, "-o", color="C2"); ax[0].axvline(1.0, ls="--", c="k", lw=1)
ax[0].set(xlabel="sigma multiplier", ylabel="signal recovery", title="Recovery vs floor (peaks at correct sigma)")
ax[1].plot(mults, meanr, "-o", label="residual mean|r| (spatial)")
ax[1].axhline(NULL, ls="--", c="k", lw=1, label="null")
ax[1].set(xlabel="sigma multiplier", ylabel="residual mean|r|", title="Spatial referee is ~flat (see next)")
ax[1].legend(fontsize=8); plt.show()

### The honest part: what the residual can and cannot tell us

Every **spatial** residual statistic (mean |r|, tails, variance, structure) is
**flat until gross over-removal** — because voxel-to-voxel correlation has *no
aggregation*, so it can't see signal that is sub-threshold *per voxel*. And that
is exactly the near-floor signal NORDIC removes. **Detecting weak signal in the
residual is the same problem, with the same information, as the original
denoising decision** — an information wall.

The most sensitive *single-dataset* referee is NORDIC's own test re-run on the
residual and **pooled over all patches** (√N_patch) — it separates *correct* from
*gross*, but is still blind to the *mild* case where recovery has already dropped.
Report it, but know its regime.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.errorbar(mults, pooled, yerr=pooled_se, fmt="-o", color="C3", label="pooled patch top/median")
# white-noise null band for the pooled statistic
null_vals = []
for _ in range(150):
    P = torch.randn(7 * 7 * nz, nt); P = P - P.mean(0, keepdim=True)
    sv = torch.linalg.svdvals(P); null_vals.append(float(sv[0] / sv.median()))
nm = float(np.mean(null_vals))
ax.axhline(nm, ls="--", c="k", lw=1, label="white null")
ax.axvline(1.0, ls=":", c="0.5"); ax.set(xlabel="sigma multiplier", ylabel="pooled top/median SV",
    title="Pooled patch-eigenvalue referee (best single-dataset stat)")
ax.legend(fontsize=8); plt.show()
print("recovery :", [round(x, 3) for x in rec])
print("pooled   :", [round(x, 4) for x in pooled], " white null ~", round(nm, 4))

## Part 2 — Real data, **0 noise volumes** (the common case)

`ds003427` sub-03 checkerboard, single-echo GE + phase, 102 vols, **no trailing
noise volumes** — so we lean on the 0-volume estimators: g-factor from the
temporal-difference surrogate (thermal is white-in-time), σ from the g-corrected
difference, and the finite-size null. A slab keeps it interactive.

In [ ]:
BENCH = Path(os.environ.get("FFS_BENCHMARK_DATA_DIR", Path.home() / ".fastfuncstuff/test_data"))
DDIR = BENCH / "ds003427-download/sub-03/func"
MAG = DDIR / "sub-03_task-checkerboard_acq-ge_run-01_bold.nii.gz"
PHA = DDIR / "sub-03_task-checkerboard_acq-ge_run-01_phase.nii.gz"

# crop a slab for interactivity (edit to taste)
XS, YS, ZS = slice(60, 180), slice(59, 179), slice(10, 20)
mag = np.asarray(nib.load(MAG).dataobj[XS, YS, ZS, :], dtype=np.float32)
pha_raw = np.asarray(nib.load(PHA).dataobj[XS, YS, ZS, :], dtype=np.float32)
pmax, pmin = pha_raw.max(), pha_raw.min()
pha = (pha_raw / (pmax - pmin) - (pmax + pmin) / (pmax - pmin) * 0.5) * (2 * np.pi)  # NORDIC phase->rad
cplx = torch.from_numpy((mag * np.exp(1j * pha)).astype(np.complex64))
del mag, pha, pha_raw
print("slab", tuple(cplx.shape))

In [ ]:
# 0-noise-volume g-factor (difference surrogate) + sigma calibration
g0_real = gf_difference(cplx, fwhm=3.0)
gcn = cplx / g0_real[..., None].clamp(min=1e-6)
# sigma from the g-corrected difference (robust: median over voxels), /sqrt(2) for the difference
dnorm = gcn[..., 1:] - gcn[..., :-1]
sig_real = float(torch.sqrt(((dnorm.real ** 2 + dnorm.imag ** 2).mean()) / 2))
print("g0 range", round(float(g0_real.min()), 2), round(float(g0_real.max()), 2), " sigma", round(sig_real, 3))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(g0_real[:, :, g0_real.shape[2] // 2].numpy().T, cmap="viridis"); ax[0].set_title("g-factor (0-vol difference)"); ax[0].axis("off")
ax[1].imshow(cplx[:, :, cplx.shape[2] // 2, 0].abs().numpy().T, cmap="gray"); ax[1].set_title("magnitude vol0"); ax[1].axis("off")
plt.show()

In [ ]:
# denoise the slab at the calibrated floor; run the referees
recon, thr, mp = denoise_hard(gcn, sigma=sig_real, kernel=(7, 7, min(7, cplx.shape[2])))
resid = gcn - recon
print(f"threshold {thr:.2f} (finite-size)  vs MP edge {mp:.2f}")

nxr, nyr, nzr = cplx.shape[:3]
iir, jjr, kkr = torch.meshgrid(torch.arange(nxr), torch.arange(nyr), torch.arange(nzr), indexing="ij")
coords_r = torch.stack([iir.ravel(), jjr.ravel(), kkr.ravel()], 1).float()
# brain-ish mask: above-median magnitude
bmag = cplx.abs().mean(-1); brain = (bmag > bmag.median()).reshape(-1)
idx_r = torch.nonzero(brain).squeeze(1)
if idx_r.numel() > 40000:
    idx_r = idx_r[torch.randperm(idx_r.numel())[:40000]]

summ = residual_whiteness(resid, idx_r, coords_r)
pm, pse = pooled_patch_topsv(resid, kernel=(7, 7, min(7, nzr)))
print(f"residual mean|r| {summ.mean_abs_r:.3f} (null {analytic_r_null(cplx.shape[-1])['mean_abs_r']:.3f})")
print(f"pooled patch top/median {pm:.4f} +/- {pse:.4f}")

## Part 3 — Toward the marginal case: **independent information**

The information wall says a single dataset can't certify the near-floor decision.
The escape is *independent* information. Two levers:

- **Multi-echo** — thermal is independent across echoes; shared-across-echoes = keep
  (already built: cross-echo rescue). This is where the optimism lives.
- **Cross-run / split-half reproducibility** (single-echo) — signal reproduces, thermal
  doesn't. The scaffold below is the workbench for the next focus: split the run,
  denoise each half, and score how much a removed/kept component's spatial loading
  reproduces. Reproducible-but-removed = over-removal you *couldn't* see any other way.

**And the sharpest caveat:** every referee here is a *global aggregate*. A single
mis-estimated patch is invisible to all of them — even multi-echo decides per patch
but can't flag a patch whose own noise estimate is wrong. Getting the noise right
**per patch** is the thing no downstream referee can rescue.

In [ ]:
def split_half_reproducibility(data_gc, sigma, kernel=(7, 7, 6)):
    """Workbench stub for the next focus. Denoise each temporal half independently;
    return the two denoised halves so you can score reproducibility of the removed
    (residual) structure. Signal reproduces across halves; thermal noise does not."""
    nt = data_gc.shape[-1]; h = nt // 2
    a, _, _ = denoise_hard(data_gc[..., :h], sigma, kernel=kernel)
    b, _, _ = denoise_hard(data_gc[..., h:2 * h], sigma, kernel=kernel)
    return a, b

# e.g. on the real slab (halved -> fewer timepoints, so the floor shifts; recalibrate for real use):
ha, hb = split_half_reproducibility(gcn, sig_real, kernel=(7, 7, min(7, cplx.shape[2])))
print("half-denoised shapes", tuple(ha.shape), tuple(hb.shape), "-- extend: score removed-component reproducibility")

### Takeaway

- g-factor is recoverable with **2, 1, or 0** noise volumes (temporal / spatial /
  difference). σ is a **separate** absolute scalar the threshold also needs.
- With g and σ right, **recovery peaks at the correct floor** — the hard threshold
  is then principled and the factor is 1.
- The residual referees catch **gross** mis-calibration only; the **mild** near-floor
  case is an information wall, escapable only with **independent** information
  (multi-echo / reproducibility) — and even those are global, so a single wrong
  patch can hide. **Get g-factor right, per patch, and everything else falls into
  place as well as it can.**